Load and explore both files

In [1]:
import pandas as pd
import numpy as np

matches = pd.read_csv('matches.csv')
deliveries = pd.read_csv('deliveries.csv')

print("Matches shape:", matches.shape)
print("Deliveries shape:", deliveries.shape)
print(matches.columns.tolist())
print(deliveries.columns.tolist())

Matches shape: (1095, 20)
Deliveries shape: (132758, 17)
['id', 'season', 'city', 'date', 'match_type', 'player_of_match', 'venue', 'team1', 'team2', 'toss_winner', 'toss_decision', 'winner', 'result', 'result_margin', 'target_runs', 'target_overs', 'super_over', 'method', 'umpire1', 'umpire2']
['match_id', 'inning', 'batting_team', 'bowling_team', 'over', 'ball', 'batter', 'bowler', 'non_striker', 'batsman_runs', 'extra_runs', 'total_runs', 'extras_type', 'is_wicket', 'player_dismissed', 'dismissal_kind', 'fielder']


Clean matches data

In [2]:
matches = matches.drop_duplicates()
matches['date'] = pd.to_datetime(matches['date'])
matches['year'] = matches['date'].dt.year

matches['toss_win_match_win'] = (
    matches['toss_winner'] == matches['winner']
).astype(int)

print(matches.isnull().sum())
matches.to_csv('matches_clean.csv', index=False)
print("Matches cleaned!")

id                       0
season                   0
city                    51
date                     0
match_type               0
player_of_match          5
venue                    0
team1                    0
team2                    0
toss_winner              0
toss_decision            0
winner                   5
result                   0
result_margin           19
target_runs              3
target_overs             3
super_over               0
method                1074
umpire1                  0
umpire2                  0
year                     0
toss_win_match_win       0
dtype: int64
Matches cleaned!


Create player batting summary

In [4]:
batting = deliveries.groupby('batter').agg(
    Total_Runs=('batsman_runs', 'sum'),
    Total_Balls=('ball', 'count'),
    Fours=('batsman_runs', lambda x: (x==4).sum()),
    Sixes=('batsman_runs', lambda x: (x==6).sum()),
    Innings=('match_id', 'nunique')
).reset_index()

batting['Strike_Rate'] = (
    batting['Total_Runs'] / batting['Total_Balls'] * 100
).round(2)

batting['Avg_per_Match'] = (
    batting['Total_Runs'] / batting['Innings']
).round(2)

batting = batting.sort_values('Total_Runs', ascending=False)
batting.to_csv('batting_summary.csv', index=False)
print("Batting summary created!")

Batting summary created!


Create team performance summary

In [5]:
team_wins = matches.groupby('winner').size().reset_index()
team_wins.columns = ['Team', 'Total_Wins']

toss_effect = matches.groupby('toss_decision')['toss_win_match_win'].mean().reset_index()
toss_effect.columns = ['Toss_Decision', 'Win_Rate']

team_wins.to_csv('team_wins.csv', index=False)
toss_effect.to_csv('toss_effect.csv', index=False)
print("Team summaries created!")

Team summaries created!


Download all clean files

In [6]:
from google.colab import files
files.download('matches_clean.csv')
files.download('batting_summary.csv')
files.download('team_wins.csv')
files.download('toss_effect.csv')
print("All files downloaded!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

All files downloaded!
